# Code Generator!

In [ ]:
# imports

import os
import requests
import re
import xml.etree.ElementTree as ET
import yaml
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display
from pathlib import Path
from typing import List



In [ ]:
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
deepseek_api_key = os.getenv('DEEPSEEK_API_KEY')
groq_api_key = os.getenv('GROQ_API_KEY')
grok_api_key = os.getenv('GROK_API_KEY')
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
if anthropic_api_key:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
else:
    print("Anthropic API Key not set (and this is optional)")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set (and this is optional)")

if deepseek_api_key:
    print(f"DeepSeek API Key exists and begins {deepseek_api_key[:3]}")
else:
    print("DeepSeek API Key not set (and this is optional)")

if groq_api_key:
    print(f"Groq API Key exists and begins {groq_api_key[:4]}")
else:
    print("Groq API Key not set (and this is optional)")

if grok_api_key:
    print(f"Grok API Key exists and begins {grok_api_key[:4]}")
else:
    print("Grok API Key not set (and this is optional)")

if openrouter_api_key:
    print(f"OpenRouter API Key exists and begins {openrouter_api_key[:3]}")
else:
    print("OpenRouter API Key not set (and this is optional)")


In [ ]:
# Connect to OpenAI client library
# A thin wrapper around calls to HTTP endpoints

openai = OpenAI()

# For Gemini, DeepSeek and Groq, we can use the OpenAI python client
# Because Google and DeepSeek have endpoints compatible with OpenAI
# And OpenAI allows you to change the base_url

anthropic_url = "https://api.anthropic.com/v1/"
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
deepseek_url = "https://api.deepseek.com"
groq_url = "https://api.groq.com/openai/v1"
grok_url = "https://api.x.ai/v1"
openrouter_url = "https://openrouter.ai/api/v1"
ollama_url = "http://localhost:11434/v1"

anthropic = OpenAI(api_key=anthropic_api_key, base_url=anthropic_url)
gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)
deepseek = OpenAI(api_key=deepseek_api_key, base_url=deepseek_url)
groq = OpenAI(api_key=groq_api_key, base_url=groq_url)
grok = OpenAI(api_key=grok_api_key, base_url=grok_url)
openrouter = OpenAI(base_url=openrouter_url, api_key=openrouter_api_key)
ollama = OpenAI(api_key="ollama", base_url=ollama_url)

claude_model = "claude-opus-4-8"
gptCodex_model = "gpt-5.3-codex"
gpt_model = "gpt-5.4"
gpt_support = "gpt-5-mini"

with open("taskDescription.txt", "r", encoding="utf-8") as f:
    task_description = f.read()
with open("systemPrompt.txt", "r", encoding="utf-8") as f:
    system_prompt = f.read()
with open("supportSystemPrompt.txt", "r", encoding="utf-8") as f:
    support_system_prompt = f.read()
with open("explorerSystemPrompt.txt", "r", encoding="utf-8") as f:
    explorer_system_prompt = f.read()
    
request = task_description

In [ ]:
def read_directory_contents(base_paths: List[str]) -> str:
    """
    Recursively reads all files under each path in base_paths and returns
    a single string containing relative file paths followed by their contents.

    :param base_paths: List of root directories to scan
    :return: Aggregated string of file paths and contents
    """
    result_parts = []

    for base_path in base_paths:
        for root, _, files in os.walk(base_path):
            for file_name in files:
                full_path = os.path.join(root, file_name)
                rel_path = os.path.relpath(full_path, base_path)

                try:
                    with open(full_path, 'r', encoding='utf-8', errors='ignore') as f:
                        content = f.read()
                except Exception as e:
                    # Skip unreadable files but record the issue
                    content = f"[Error reading file: {e}]"

                result_parts.append(
                    f"File directory: {rel_path}\n\n"
                    f"{content}\n"
                )

    return "\n".join(result_parts)

In [ ]:

allCodeSimple = read_directory_contents(["C:/Users/ME36352/MyFolder/REPOS/ACE/services-integration-finance-accounting-assistance-invoices/apps/services-integration-finance-accounting-assistance-invoices"])

In [ ]:
def collect_service_with_dependencies(repository_path: str, service_name: str) -> str:
    """
    Collects all files from a service and its transitive dependencies.
    Handles Maven (pom.xml) + Eclipse (.project) based dependency resolution.
    """

    repository = Path(repository_path).resolve()
    service_root = (repository / service_name).resolve()

    if not service_root.exists():
        raise FileNotFoundError(f"Service '{service_name}' not found in repository.")

    # ------------------------------------------------------------------
    # Ignore rules
    # ------------------------------------------------------------------

    GIT_FILES = {".gitignore", ".gitattributes", ".gitmodules"}

    BINARY_EXTENSIONS = {
        ".jar", ".class", ".exe", ".dll", ".zip", ".war", ".ear",
        ".png", ".jpg", ".jpeg", ".gif", ".bmp", ".ico", ".pdf",
        ".so", ".o", ".obj", ".a", ".lib"
    }

    def should_skip(path: Path) -> bool:
        return (
            ".git" in path.parts or
            path.name in GIT_FILES
        )

    # ------------------------------------------------------------------
    # Index repository once
    # ------------------------------------------------------------------

    artifact_to_dirs = {}      # artifactId -> possible project dirs
    project_files = []         # all .project files

    for file in repository.rglob("*"):
        if should_skip(file):
            continue

        if not file.is_file():
            continue

        # ---- Maven artifacts ----
        if file.name == "pom.xml":
            try:
                tree = ET.parse(file)
                root = tree.getroot()

                if root.tag.startswith("{"):
                    uri = root.tag.split("}")[0][1:]
                    ns = {"m": uri}
                    artifact = root.find("m:artifactId", ns)
                else:
                    artifact = root.find("artifactId")

                if artifact is not None and artifact.text:
                    artifact_to_dirs.setdefault(artifact.text.strip(), []).append(file.parent.resolve())

            except Exception:
                pass

        # ---- Eclipse project marker ----
        elif file.name == ".project":
            project_files.append(file.resolve())

    # ------------------------------------------------------------------
    # Resolve .project belonging to a directory
    # ------------------------------------------------------------------

    def find_project_file(project_dir: Path):
        project_dir = project_dir.resolve()

        for pf in project_files:
            try:
                pf.relative_to(project_dir)
                return pf
            except Exception:
                continue

        return None

    # ------------------------------------------------------------------
    # Dependency parsing
    # ------------------------------------------------------------------

    def extract_dependencies(project_file: Path):
        if not project_file:
            return []

        deps = []

        try:
            tree = ET.parse(project_file)
            root = tree.getroot()

            projects = root.find("projects")
            if projects is None:
                return deps

            for p in projects.findall("project"):
                if p.text:
                    deps.append(p.text.strip())

        except Exception:
            pass

        return deps

    # ------------------------------------------------------------------
    # Output handling
    # ------------------------------------------------------------------

    output = []
    visited = set()

    # ------------------------------------------------------------------
    # File reader
    # ------------------------------------------------------------------

    def read_files(base_dir: Path):

        for file in sorted(base_dir.rglob("*")):

            if should_skip(file):
                continue

            if not file.is_file():
                continue

            rel = file.relative_to(repository)

            # ---- binary handling ----
            if file.suffix.lower() in BINARY_EXTENSIONS:
                if file.suffix.lower() == ".jar":
                    msg = "--- JAR CONTENT NOT READABLE ---"
                elif file.suffix.lower() == ".class":
                    msg = "--- JAVA CLASS NOT READABLE ---"
                else:
                    msg = "--- BINARY FILE NOT READABLE ---"

                output.append(
                    f"File directory: {rel}\n\n{msg}\n\n" + "=" * 80 + "\n\n"
                )
                continue

            # ---- text files ----
            try:
                content = file.read_text(encoding="utf-8", errors="ignore")
            except Exception:
                continue

            output.append(
                f"File directory: {rel}\n\n{content}\n\n" + "=" * 80 + "\n\n"
            )

    # ------------------------------------------------------------------
    # Project resolver (core fix: now correctly walks dependencies)
    # ------------------------------------------------------------------

    def resolve_project(project_dir: Path):

        project_dir = project_dir.resolve()

        if project_dir in visited:
            return

        visited.add(project_dir)

        # 1. read files in this project
        read_files(project_dir)

        # 2. get .project
        project_file = find_project_file(project_dir)

        # 3. dependencies from .project
        deps = extract_dependencies(project_file)

        # 4. resolve dependencies via artifactId
        for dep in deps:
            candidate_dirs = artifact_to_dirs.get(dep, [])

            for dep_dir in candidate_dirs:
                resolve_project(dep_dir)

    # ------------------------------------------------------------------

    resolve_project(service_root)

    return "".join(output)

In [ ]:
repository = r"C:/Users/ME36352/MyFolder/REPOS/ACE"
allCode = collect_service_with_dependencies(repository, "services-integration-finance-accounting-assistance-invoices")

In [ ]:
 
def tree_to_yaml(root_path: str) -> str:
    """
    Build a recursive directory + file tree and return it as YAML.
    Directories are nested mappings; files are keys with null values.
    """
    root = Path(root_path)

    def build_tree(path: Path) -> dict:
        tree = {}
        # directories first, then files; both sorted alphabetically
        items = sorted(path.iterdir(), key=lambda p: (p.is_file(), p.name.lower()))
        for item in items:
            if item.is_dir() and item.name != ".git":
                tree[item.name] = build_tree(item)
            else:
                tree[item.name] = None
        return tree

    data = {root.name: build_tree(root)}
    return yaml.safe_dump(data, sort_keys=False, default_flow_style=False)

In [ ]:
projectTree = tree_to_yaml("C:/Users/ME36352/MyFolder/REPOS/ACE")

In [ ]:
def call_gpt(chat, dev_prompt, llm_model, reasoning):
    messages = [
       {"role": "system", "content": dev_prompt},
        {"role": "user", "content": chat}
    ]
    
    response = openai.chat.completions.create(model=llm_model, messages=messages, reasoning_effort=reasoning)
    return response.choices[0].message.content

In [ ]:
explorerRequest = projectTree + "\n\n" + allCodeSimple
explorerResponse = call_gpt(explorerRequest, explorer_system_prompt, gpt_support, "medium")
print(explorerResponse)

In [ ]:
def process_pipe_values(text):
    parts = text.split("|")
    response = "" 
    for part in parts:
        if part:  # ignores empty values from leading/trailing "|"
            response = response + collect_service_with_dependencies(repository, part)
    return response        

In [ ]:
allCode = allCode + "\n" + process_pipe_values(explorerResponse)

In [ ]:
def save_string_to_txtFile(content: str, filename: str):

    filename += ".txt"
    output_path =  Path.cwd() / filename
    output_path.write_text(content, encoding="utf-8")


In [ ]:
save_string_to_txtFile(allCode, "serviceAndDependencies")

In [ ]:
system_prompt = system_prompt + allCode

In [ ]:
gptSupport = call_gpt(request, support_system_prompt, gpt_support, "high")
request = "ORIGINAL REQUEST: " + request + "\n\nANALYZED REQUEST BY LLM:\n" + gptSupport 
save_string_to_txtFile(request, "taskDescriptionEnhanced")

In [ ]:
gptResponse = call_gpt(request, system_prompt, gpt_model, "high")
save_string_to_txtFile(gptResponse, "gptResponse")
print(gptResponse)

In [ ]:
def call_claude(chat):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": chat}
    ]

    response = anthropic.chat.completions.create(
        model=claude_model,
        messages=messages,
        max_tokens=128000,                # REQUIRED for Anthropic
        extra_body={
            "thinking": {
                "type": "enabled",
                "budget_tokens": 126000   # must be < max_tokens
            }
        }
    )
    return response.choices[0].message.content

In [ ]:
claudeResponse = call_claude(request)
save_string_to_txtFile(claudeResponse, "claudeResponse")
print(claudeResponse)

In [ ]:
def call_gptCodex(prompt: str):
    response = openai.responses.create(
        model=gptCodex_model,
        reasoning={"effort": "high"},
        instructions=system_prompt,
        input=prompt,
    )

    return response.output_text

In [ ]:
gptCodexResponse = call_gptCodex(request)
save_string_to_txtFile(gptCodexResponse, "gptCodexResponse")
print(gptCodexResponse)

In [ ]:
def apply_llm_response(directory_path: str, llm_response: str) -> None:
    """
    Parses an LLM response of the format:

    --FN--FileName.ext--FN--
    --FC--FileContent--FC--

    ...

    ---COMMENT---
    comment text
    ---COMMENT---

    Creates all files in the specified directory and writes the comment
    into comment.txt.
    """

    target_dir = Path(directory_path)
    target_dir.mkdir(parents=True, exist_ok=True)

    # Extract comment
    comment_pattern = re.compile(
        r"---COMMENT---\s*(.*?)\s*---COMMENT---",
        re.DOTALL
    )

    comment_match = comment_pattern.search(llm_response)
    comment_text = comment_match.group(1).strip() if comment_match else ""

    # Write comment.txt
    (target_dir / "comment.txt").write_text(
        comment_text,
        encoding="utf-8"
    )

    # Remove comment section before parsing files
    content_without_comment = comment_pattern.sub("", llm_response)

    # Extract files
    file_pattern = re.compile(
        r"--FN--\s*(.*?)\s*--FN--\s*"
        r"--FC--\s*(.*?)\s*--FC--",
        re.DOTALL
    )

    for match in file_pattern.finditer(content_without_comment):
        filename = match.group(1).strip()
        file_content = match.group(2)

        file_path = target_dir / filename

        # Create parent directories if needed
        file_path.parent.mkdir(parents=True, exist_ok=True)

        file_path.write_text(file_content, encoding="utf-8")

In [ ]:
apply_llm_response("C:/Users/ME36352/MyFolder/AI/Repos/AIRepo/codeGenerator/outputClaude", claudeResponse)

In [ ]:
apply_llm_response("C:/Users/ME36352/MyFolder/AI/Repos/AIRepo/codeGenerator/outputGPT", gptResponse)

In [ ]:
apply_llm_response("C:/Users/ME36352/MyFolder/AI/Repos/AIRepo/codeGenerator/outputGPTCodex", gptCodexResponse)